# 01 — Clean Corpus

Reads raw JSONL dumps and produces clean, canonical JSONL files used by all downstream notebooks.

**Inputs:**
- `data/raw/r_gradadmissions_posts.jsonl`
- `data/raw/r_gradadmissions_comments.jsonl`

**Outputs:**
- `data/processed_v2/posts_clean.jsonl` — one post per line: `id, author, created_dt, clean_text, score, num_comments`
- `data/processed_v2/comments_clean.jsonl` — one comment per line: `id, author, created_dt, post_id, clean_text, score`

**Cleaning steps applied:**
1. Date/author validation — parse `created_utc`, drop deleted/removed/null authors and bodies
2. Dedup & bot filtering — deduplicate on `id`, drop `AutoModerator` and bot accounts
3. Text normalization — lowercase, strip URLs, strip non-alpha, collapse whitespace → `clean_text`
4. Comment→post mapping — derive `post_id` from `link_id` (strip `t3_` prefix)

In [1]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT         = Path('..').resolve()
RAW_DIR      = ROOT / 'data' / 'raw'
OUT_DIR      = ROOT / 'data' / 'processed_v2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

POSTS_IN     = RAW_DIR / 'r_gradadmissions_posts.jsonl'
COMMENTS_IN  = RAW_DIR / 'r_gradadmissions_comments.jsonl'
POSTS_OUT    = OUT_DIR / 'posts_clean.jsonl'
COMMENTS_OUT = OUT_DIR / 'comments_clean.jsonl'

print('Posts in:   ', POSTS_IN)
print('Comments in:', COMMENTS_IN)
print('Out dir:    ', OUT_DIR)

Posts in:    /media/ayush/F/Coding/CS598_Research_Project/data/raw/r_gradadmissions_posts.jsonl
Comments in: /media/ayush/F/Coding/CS598_Research_Project/data/raw/r_gradadmissions_comments.jsonl
Out dir:     /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2


## Helper functions

In [2]:
_URL_RE   = re.compile(r'https?://\S+|www\.\S+')
_NONALPHA = re.compile(r'[^a-zA-Z\s]')
_SPACES   = re.compile(r'\s+')
_BOT_RE   = re.compile(r'(?i)bot$')

DELETED_VALS = {'[deleted]', '[removed]', 'None', None}
BOT_NAMES    = {'AutoModerator'}


def parse_utc(val) -> str | None:
    """Return ISO datetime string (UTC) from Unix timestamp or ISO string."""
    if val is None:
        return None
    try:
        return datetime.fromtimestamp(float(val), tz=timezone.utc).isoformat()
    except (TypeError, ValueError):
        pass
    try:
        return datetime.fromisoformat(str(val)).isoformat()
    except ValueError:
        return None


def clean_text(raw: str) -> str:
    """Lowercase, strip URLs, strip non-alpha, collapse whitespace."""
    t = _URL_RE.sub(' ', raw.lower())
    t = _NONALPHA.sub(' ', t)
    return _SPACES.sub(' ', t).strip()


def is_bot(author: str) -> bool:
    return author in BOT_NAMES or bool(_BOT_RE.search(author))


def is_deleted(val) -> bool:
    return val in DELETED_VALS or (
        isinstance(val, str) and val.strip() in {'[deleted]', '[removed]', ''}
    )


print('Helpers defined.')

Helpers defined.


## 1) Clean posts

In [3]:
counts_posts = {}

raw_posts = []
with open(POSTS_IN) as f:
    for line in f:
        line = line.strip()
        if line:
            raw_posts.append(json.loads(line))

counts_posts['loaded'] = len(raw_posts)
print(f"Loaded:                       {counts_posts['loaded']:>7,}")

Loaded:                        90,680


In [4]:
# Step 1a: date/author/body validation
valid_posts = []
for r in raw_posts:
    dt = parse_utc(r.get('created_utc'))
    if dt is None:
        continue
    author = r.get('author')
    if is_deleted(author):
        continue
    selftext = r.get('selftext', '') or ''
    title    = r.get('title', '')    or ''
    combined = (title + ' ' + selftext).strip()
    if is_deleted(selftext) or len(combined) <= 5:
        continue
    r['_dt']       = dt
    r['_combined'] = combined
    valid_posts.append(r)

counts_posts['after_date_author'] = len(valid_posts)
print(f"After date/author validation: {counts_posts['after_date_author']:>7,}")

After date/author validation:  78,986


In [5]:
# Step 1b: dedup on id (keep first occurrence)
seen_ids = set()
deduped  = []
for r in valid_posts:
    if r['id'] not in seen_ids:
        seen_ids.add(r['id'])
        deduped.append(r)

counts_posts['after_dedup'] = len(deduped)
print(f"After dedup:                  {counts_posts['after_dedup']:>7,}")

After dedup:                   78,986


In [6]:
# Step 1c: bot filtering
no_bots = [r for r in deduped if not is_bot(str(r.get('author', '')))]

counts_posts['after_bot_filter'] = len(no_bots)
print(f"After bot filter:             {counts_posts['after_bot_filter']:>7,}")

After bot filter:              78,962


In [7]:
# Step 1d: text normalization → build output records
clean_posts = []
for r in no_bots:
    ct = clean_text(r['_combined'])
    if len(ct) < 5:
        continue
    clean_posts.append({
        'id':           r['id'],
        'author':       r['author'],
        'created_dt':   r['_dt'],
        'clean_text':   ct,
        'score':        r.get('score', 0),
        'num_comments': r.get('num_comments', 0),
    })

counts_posts['final'] = len(clean_posts)
print(f"Final clean posts:            {counts_posts['final']:>7,}")

Final clean posts:             78,961


In [8]:
print('\n--- Posts cleaning summary ---')
for step, n in counts_posts.items():
    print(f'  {step:<28} {n:>7,}')

print('\nSample (5 rows):')
pd.DataFrame(clean_posts[:5])[['id', 'author', 'created_dt', 'score', 'num_comments', 'clean_text']] \
  .assign(clean_text=lambda d: d['clean_text'].str[:80])


--- Posts cleaning summary ---
  loaded                        90,680
  after_date_author             78,986
  after_dedup                   78,986
  after_bot_filter              78,962
  final                         78,961

Sample (5 rows):


,id,author,created_dt,score,num_comments,clean_text
0,15ex9yw,ArmLongjumping3965,2023-08-01T00:39:02+00:00,1,0,master s in engineering holder looking to lear...
1,15ezubg,Sherlock-1899,2023-08-01T02:36:37+00:00,2,0,can i get into a good college with backlogs i ...
2,15f0g9d,AromaticAd6947,2023-08-01T03:05:48+00:00,1,4,should i retake gre i got verbal quant and ana...
3,15f1bm3,Lab_Rat13,2023-08-01T03:49:07+00:00,2,2,am i eligible for a masters in computer scienc...
4,15f1njp,Jared_9000,2023-08-01T04:05:22+00:00,3,3,looking for any and all advice for a prospecti...


## 2) Clean comments

In [9]:
counts_comments = {}

raw_comments = []
with open(COMMENTS_IN) as f:
    for line in f:
        line = line.strip()
        if line:
            raw_comments.append(json.loads(line))

counts_comments['loaded'] = len(raw_comments)
print(f"Loaded:                       {counts_comments['loaded']:>7,}")

Loaded:                       500,688


In [10]:
# Step 2a: date/author/body validation
valid_comments = []
for r in raw_comments:
    dt = parse_utc(r.get('created_utc'))
    if dt is None:
        continue
    author = r.get('author')
    if is_deleted(author):
        continue
    body = r.get('body', '') or ''
    if is_deleted(body) or len(body.strip()) <= 5:
        continue
    link_id = r.get('link_id', '')
    if not link_id:
        continue
    r['_dt'] = dt
    valid_comments.append(r)

counts_comments['after_date_author'] = len(valid_comments)
print(f"After date/author validation: {counts_comments['after_date_author']:>7,}")

After date/author validation: 472,113


In [11]:
# Step 2b: dedup on id
seen_ids = set()
deduped  = []
for r in valid_comments:
    if r['id'] not in seen_ids:
        seen_ids.add(r['id'])
        deduped.append(r)

counts_comments['after_dedup'] = len(deduped)
print(f"After dedup:                  {counts_comments['after_dedup']:>7,}")

After dedup:                  472,113


In [12]:
# Step 2c: bot filtering
no_bots = [r for r in deduped if not is_bot(str(r.get('author', '')))]

counts_comments['after_bot_filter'] = len(no_bots)
print(f"After bot filter:             {counts_comments['after_bot_filter']:>7,}")

After bot filter:             470,074


In [13]:
# Step 2d: text normalization + comment→post mapping
clean_comments = []
for r in no_bots:
    ct = clean_text(r['body'])
    if len(ct) < 5:
        continue
    link_id = r.get('link_id', '')
    post_id = link_id.removeprefix('t3_') if link_id else ''
    clean_comments.append({
        'id':         r['id'],
        'author':     r['author'],
        'created_dt': r['_dt'],
        'post_id':    post_id,
        'clean_text': ct,
        'score':      r.get('score', 0),
    })

counts_comments['final'] = len(clean_comments)
print(f"Final clean comments:         {counts_comments['final']:>7,}")

Final clean comments:         467,986


In [14]:
print('\n--- Comments cleaning summary ---')
for step, n in counts_comments.items():
    print(f'  {step:<28} {n:>7,}')

print('\nSample (5 rows) — check post_id populated:')
pd.DataFrame(clean_comments[:5])[['id', 'author', 'created_dt', 'post_id', 'score', 'clean_text']] \
  .assign(clean_text=lambda d: d['clean_text'].str[:60])


--- Comments cleaning summary ---
  loaded                       500,688
  after_date_author            472,113
  after_dedup                  472,113
  after_bot_filter             470,074
  final                        467,986

Sample (5 rows) — check post_id populated:


,id,author,created_dt,post_id,score,clean_text
0,jua0cim,Nay_Nay_Jonez,2023-08-01T00:15:02+00:00,15eotze,5,email is totally fine also include a list of s...
1,jua0vwg,Particular_Day_1815,2023-08-01T00:18:54+00:00,15c3104,1,i was planning to apply for winter intake but ...
2,jua33tq,Bumblby-Life,2023-08-01T00:34:55+00:00,15eprle,2,how early if applications are due in december ...
3,jua3xng,DivineCherriBlossom,2023-08-01T00:40:52+00:00,15et8et,1,thank you so much
4,jua7bg3,tossin_glitter,2023-08-01T01:05:32+00:00,1381lj2,1,sorry for such a late response but figured i d...


In [15]:
# Verify post_id→post cross-reference
post_ids_set      = {p['id'] for p in clean_posts}
comment_post_ids  = {c['post_id'] for c in clean_comments}
overlap           = len(comment_post_ids & post_ids_set)
print(f"Unique post_ids in comments:   {len(comment_post_ids):>6,}")
print(f"Of those found in posts_clean: {overlap:>6,}  ({100*overlap/max(len(comment_post_ids),1):.1f}%)")

Unique post_ids in comments:   62,134
Of those found in posts_clean: 55,296  (89.0%)


## 3) Write output files

> **Approval gate:** Review the summaries and samples above before running this cell.

In [16]:
with open(POSTS_OUT, 'w') as f:
    for rec in clean_posts:
        f.write(json.dumps(rec) + '\n')
print(f'Wrote {len(clean_posts):,} posts → {POSTS_OUT}')

with open(COMMENTS_OUT, 'w') as f:
    for rec in clean_comments:
        f.write(json.dumps(rec) + '\n')
print(f'Wrote {len(clean_comments):,} comments → {COMMENTS_OUT}')

Wrote 78,961 posts → /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/posts_clean.jsonl
Wrote 467,986 comments → /media/ayush/F/Coding/CS598_Research_Project/data/processed_v2/comments_clean.jsonl
